# MIC Creep — Data Diversity & Filtering Explorer

**Goal:** Understand how censoring and MIC diversity are distributed across countries and continents. Experiment with filtering strategies to make the training set more informative.

**Upload required:** `X_train.parquet` and `y_train.parquet` from `data/processed/kpneumoniae/` or `abaumannii/`.

> These processed files contain no raw isolate identifiers — only features and log2(MIC) targets. Handle with care; do not share publicly.

In [ ]:
!pip install pyarrow seaborn -q

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (14, 5)})
sns.set_theme(style='whitegrid', palette='muted')

LOG2_R     = np.log2(8.0)    # EUCAST R breakpoint
LOG2_FLOOR = np.log2(0.06)   # left censoring limit
LOG2_CEIL  = np.log2(32.0)   # right censoring limit

## 1. Load Data

# --- Option A: Google Drive (recommended) ---
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/mic_creep_data/'  # adjust path if needed
X = pd.read_parquet(DATA_DIR + 'X_train.parquet')
y = pd.read_parquet(DATA_DIR + 'y_train.parquet').squeeze()

print(f'X shape: {X.shape}')
print(f'y range: [{y.min():.2f}, {y.max():.2f}] log2 = [{2**y.min():.3f}, {2**y.max():.1f}] mg/L')
print(f'Columns sample: {list(X.columns[:10])}')

In [ ]:
from google.colab import files
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (14, 5)})
sns.set_theme(style='whitegrid', palette='muted')

LOG2_R     = np.log2(8.0)
LOG2_FLOOR = np.log2(0.06)
LOG2_CEIL  = np.log2(32.0)

print('Select X_train.parquet and y_train.parquet when the dialog opens')
uploaded = files.upload()

X = pd.read_parquet(io.BytesIO(uploaded['X_train.parquet']))
y = pd.read_parquet(io.BytesIO(uploaded['y_train.parquet'])).squeeze()

print(f'X shape: {X.shape}')
print(f'y range: [{y.min():.2f}, {y.max():.2f}] log2 = [{2**y.min():.3f}, {2**y.max():.1f}] mg/L')


## 2. Reconstruct Country & Build Working DataFrame

In [ ]:
# Reverse the OHE: find which ctry_ column is 1 for each row
ctry_cols = [c for c in X.columns if c.startswith('ctry_')]
print(f'{len(ctry_cols)} country columns found')

ctry_sum = X[ctry_cols].sum(axis=1)
country_series = X[ctry_cols].idxmax(axis=1).str.replace('ctry_', '', regex=False)
# Rows where all ctry_ = 0 are the OHE reference country (one was dropped)
country_series[ctry_sum == 0] = 'Reference (dropped)'

df = pd.DataFrame({
    'country': country_series,
    'year': X['year'].astype(int),
    'is_censored': X['is_censored'].astype(int),
    'log2_mic': y.values,
    'mic_mgL': 2 ** y.values,
    'resistant': (y.values >= LOG2_R).astype(int),
})

# Add resistance genes if present
gene_cols = [c for c in X.columns if c.endswith('_pos')]
for g in gene_cols:
    df[g] = X[g].values

print(df.describe())
print(f'\nCountries: {df["country"].nunique()}')
print(f'Years: {df["year"].min()} - {df["year"].max()}')
print(f'Overall censoring: {df["is_censored"].mean():.1%}')
print(f'Overall resistant: {df["resistant"].mean():.1%}')

## 3. Continent Mapping

In [ ]:
CONTINENT = {
    # North America
    'United States': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
    'Guatemala': 'North America', 'Dominican Republic': 'North America',
    'Costa Rica': 'North America', 'Jamaica': 'North America', 'Panama': 'North America',
    # South America
    'Brazil': 'South America', 'Argentina': 'South America', 'Colombia': 'South America',
    'Chile': 'South America', 'Peru': 'South America', 'Venezuela': 'South America',
    'Ecuador': 'South America', 'Bolivia': 'South America', 'Uruguay': 'South America',
    # Europe (West)
    'Germany': 'Europe (W)', 'France': 'Europe (W)', 'United Kingdom': 'Europe (W)',
    'Italy': 'Europe (W)', 'Spain': 'Europe (W)', 'Portugal': 'Europe (W)',
    'Belgium': 'Europe (W)', 'Netherlands': 'Europe (W)', 'Austria': 'Europe (W)',
    'Switzerland': 'Europe (W)', 'Sweden': 'Europe (W)', 'Norway': 'Europe (W)',
    'Denmark': 'Europe (W)', 'Finland': 'Europe (W)', 'Ireland': 'Europe (W)',
    # Europe (East/Central)
    'Poland': 'Europe (E)', 'Czech Republic': 'Europe (E)', 'Hungary': 'Europe (E)',
    'Romania': 'Europe (E)', 'Bulgaria': 'Europe (E)', 'Serbia': 'Europe (E)',
    'Croatia': 'Europe (E)', 'Slovakia': 'Europe (E)', 'Slovenia': 'Europe (E)',
    'Lithuania': 'Europe (E)', 'Latvia': 'Europe (E)', 'Estonia': 'Europe (E)',
    'Ukraine': 'Europe (E)', 'Russia': 'Europe (E)', 'Greece': 'Europe (E)',
    'Turkey': 'Europe (E)', 'Iceland': 'Europe (E)',
    # Middle East
    'Saudi Arabia': 'Middle East', 'Israel': 'Middle East', 'UAE': 'Middle East',
    'Jordan': 'Middle East', 'Lebanon': 'Middle East', 'Kuwait': 'Middle East',
    'Bahrain': 'Middle East', 'Oman': 'Middle East', 'Qatar': 'Middle East',
    'Iraq': 'Middle East', 'Iran': 'Middle East', 'Egypt': 'Middle East',
    # Asia Pacific
    'China': 'Asia Pacific', 'Japan': 'Asia Pacific', 'South Korea': 'Asia Pacific',
    'Taiwan': 'Asia Pacific', 'India': 'Asia Pacific', 'Thailand': 'Asia Pacific',
    'Malaysia': 'Asia Pacific', 'Indonesia': 'Asia Pacific', 'Philippines': 'Asia Pacific',
    'Vietnam': 'Asia Pacific', 'Singapore': 'Asia Pacific', 'Pakistan': 'Asia Pacific',
    'Bangladesh': 'Asia Pacific', 'Sri Lanka': 'Asia Pacific', 'Hong Kong': 'Asia Pacific',
    'Australia': 'Asia Pacific', 'New Zealand': 'Asia Pacific',
    # Africa
    'South Africa': 'Africa', 'Morocco': 'Africa', 'Nigeria': 'Africa',
    'Kenya': 'Africa', 'Tunisia': 'Africa', 'Algeria': 'Africa',
    'Reference (dropped)': 'Unknown',
}

df['continent'] = df['country'].map(CONTINENT).fillna('Other')

unmapped = df[df['continent'] == 'Other']['country'].unique()
if len(unmapped) > 0:
    print('Unmapped countries (add to dict above):', unmapped)
else:
    print('All countries mapped.')

print('\nIsolates per continent:')
print(df.groupby('continent').size().sort_values(ascending=False).to_string())

## 4. Censoring Landscape

In [ ]:
# Country-level stats
ctry_stats = df.groupby('country').agg(
    n=('log2_mic', 'count'),
    censoring_rate=('is_censored', 'mean'),
    mic_mean=('log2_mic', 'mean'),
    mic_p50=('log2_mic', 'median'),
    mic_p90=('log2_mic', lambda x: np.percentile(x, 90)),
    resistant_pct=('resistant', 'mean'),
    mic_iqr=('log2_mic', lambda x: np.percentile(x, 75) - np.percentile(x, 25)),
).reset_index()
ctry_stats['continent'] = ctry_stats['country'].map(CONTINENT).fillna('Other')

print('Top 20 countries by censoring rate:')
print(ctry_stats.sort_values('censoring_rate', ascending=False)
    [['country', 'n', 'censoring_rate', 'mic_p90', 'resistant_pct']]
    .head(20).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: censoring rate by continent (boxplot)
cont_order = (ctry_stats.groupby('continent')['censoring_rate']
              .median().sort_values(ascending=False).index.tolist())
ctry_stats.boxplot(column='censoring_rate', by='continent', ax=axes[0],
                   grid=False, rot=30)
axes[0].set_title('Censoring rate by continent')
axes[0].set_xlabel('')
axes[0].set_ylabel('Fraction of isolates at panel floor')
plt.sca(axes[0])
plt.xticks(rotation=30, ha='right')

# Right: MIC IQR (diversity) vs censoring rate, coloured by continent
palette = sns.color_palette('tab10', n_colors=ctry_stats['continent'].nunique())
cont_list = ctry_stats['continent'].unique()
color_map = dict(zip(cont_list, palette))

for cont, grp in ctry_stats.groupby('continent'):
    axes[1].scatter(grp['censoring_rate'], grp['mic_iqr'],
                    label=cont, alpha=0.7, s=grp['n'] / grp['n'].max() * 200 + 20,
                    color=color_map[cont])

axes[1].set_xlabel('Censoring rate')
axes[1].set_ylabel('MIC IQR (log2) — higher = more diverse')
axes[1].set_title('MIC diversity vs censoring rate (bubble = n isolates)')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[1].axvline(0.5, color='red', lw=1, ls='--', label='50% censoring')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# MIC distribution by continent - violin
fig, ax = plt.subplots(figsize=(14, 5))
cont_order_mic = (df.groupby('continent')['log2_mic']
                  .median().sort_values().index.tolist())
sns.violinplot(data=df, x='continent', y='log2_mic', order=cont_order_mic,
               inner='quartile', cut=0, ax=ax)
ax.axhline(LOG2_R, color='red', lw=1.5, ls='--', label='EUCAST R (MIC=8)')
ax.axhline(LOG2_FLOOR, color='gray', lw=1, ls=':', label='Panel floor (MIC=0.06)')
ax.set_xlabel('')
ax.set_ylabel('log2(MIC)')
ax.set_title('MIC distribution by continent (train set)')
ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 5. Country-Level Heatmap: Censoring x Resistant %

In [ ]:
# Only countries with >= 100 isolates
big_countries = ctry_stats[ctry_stats['n'] >= 100].copy()
big_countries = big_countries.sort_values(['continent', 'censoring_rate'])

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(big_countries) * 0.25)))

for ax, col, title, cmap in zip(
    axes,
    ['censoring_rate', 'resistant_pct'],
    ['Censoring rate', 'Resistant %'],
    ['YlOrRd', 'Blues'],
):
    pivot = big_countries.set_index('country')[[col]]
    sns.heatmap(pivot, ax=ax, cmap=cmap, vmin=0, vmax=1,
                annot=True, fmt='.0%', linewidths=0.3, linecolor='white',
                cbar_kws={'shrink': 0.6})
    ax.set_title(title)
    ax.set_ylabel('')
    ax.set_xlabel('')

plt.suptitle('Countries with >= 100 isolates', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Temporal Trends by Continent

In [ ]:
# MIC90 trend per continent over years
trend = (df.groupby(['year', 'continent'])['log2_mic']
         .quantile(0.9).reset_index()
         .rename(columns={'log2_mic': 'mic90_log2'}))

fig, ax = plt.subplots(figsize=(14, 5))
for cont, grp in trend.groupby('continent'):
    grp = grp.sort_values('year')
    ax.plot(grp['year'], grp['mic90_log2'], marker='o', ms=4, label=cont)

ax.axhline(LOG2_R, color='red', lw=1.5, ls='--', label='EUCAST R')
ax.axvline(2018.5, color='gray', lw=1, ls=':', label='Train/Test split')
ax.set_xlabel('Year')
ax.set_ylabel('MIC90 (log2)')
ax.set_title('MIC90 trend by continent (train set)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Censoring rate trend per continent
cens_trend = (df.groupby(['year', 'continent'])['is_censored']
              .mean().reset_index())

fig, ax = plt.subplots(figsize=(14, 4))
for cont, grp in cens_trend.groupby('continent'):
    grp = grp.sort_values('year')
    ax.plot(grp['year'], grp['is_censored'], marker='o', ms=4, label=cont)

ax.axvline(2018.5, color='gray', lw=1, ls=':', label='Train/Test split')
ax.set_xlabel('Year')
ax.set_ylabel('Censoring rate')
ax.set_title('Censoring rate over time by continent')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Filtering Experiments

Key question: if we remove high-censoring countries (where almost everything is at the floor), does the training set become more informative for the model without losing too much data?

In [ ]:
def filter_summary(mask, label):
    sub = df[mask]
    return {
        'Filter': label,
        'N train': len(sub),
        'N kept %': f"{len(sub)/len(df):.1%}",
        'Censoring': f"{sub['is_censored'].mean():.1%}",
        'Resistant %': f"{sub['resistant'].mean():.1%}",
        'MIC IQR (log2)': f"{sub['log2_mic'].quantile(0.75) - sub['log2_mic'].quantile(0.25):.3f}",
        'MIC90 (log2)': f"{sub['log2_mic'].quantile(0.9):.3f}",
        'N countries': sub['country'].nunique(),
    }

# Country-level censoring rates for filtering
ctry_cens = df.groupby('country')['is_censored'].mean().rename('ctry_cens_rate')
df['ctry_cens_rate'] = df['country'].map(ctry_cens)

experiments = [
    (np.ones(len(df), dtype=bool),                      'No filter (baseline)'),
    (df['ctry_cens_rate'] < 0.90,                       'Drop countries >90% censored'),
    (df['ctry_cens_rate'] < 0.75,                       'Drop countries >75% censored'),
    (df['ctry_cens_rate'] < 0.50,                       'Drop countries >50% censored'),
    (~df['continent'].isin(['Other', 'Unknown']),        'Drop unmapped continents'),
    (df['continent'].isin(['Europe (E)', 'Asia Pacific', 'Middle East']),
                                                         'Keep high-resistance continents only'),
    (df['resistant'] == 1,                               'Resistant isolates only'),
]

summary = pd.DataFrame([filter_summary(mask, label) for mask, label in experiments])
print(summary.to_string(index=False))

In [ ]:
# Visual: MIC distributions under different filters
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for ax, (mask, label) in zip(axes, experiments[1:]):
    sub = df[mask]['log2_mic']
    ax.hist(sub, bins=40, color='steelblue', alpha=0.75, edgecolor='white', lw=0.3)
    ax.axvline(LOG2_R, color='red', lw=1.5, ls='--')
    ax.axvline(LOG2_FLOOR, color='gray', lw=1, ls=':')
    ax.set_title(f'{label}\n(n={len(sub):,}, cens={mask.sum()/len(df):.0%} kept)', fontsize=9)
    ax.set_xlabel('log2(MIC)')

plt.suptitle('MIC distributions under different training set filters\n'
             '(red dashed = EUCAST R, gray dotted = panel floor)', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Censoring by Country — Full Ranking

In [ ]:
ranked = (ctry_stats
    .sort_values('censoring_rate', ascending=True)
    .reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(ranked) * 0.22)))

# Left: censoring rate
colors = ['#d73027' if r >= 0.75 else '#fc8d59' if r >= 0.50 else '#91bfdb'
          for r in ranked['censoring_rate']]
axes[0].barh(ranked['country'], ranked['censoring_rate'], color=colors)
axes[0].axvline(0.75, color='red', lw=1, ls='--')
axes[0].axvline(0.50, color='orange', lw=1, ls='--')
axes[0].set_xlabel('Censoring rate')
axes[0].set_title('Censoring rate by country')

patches = [
    mpatches.Patch(color='#d73027', label='>75% censored (consider drop)'),
    mpatches.Patch(color='#fc8d59', label='50-75% censored (caution)'),
    mpatches.Patch(color='#91bfdb', label='<50% censored (informative)'),
]
axes[0].legend(handles=patches, loc='lower right', fontsize=8)

# Right: n isolates
axes[1].barh(ranked['country'], ranked['n'], color='#4393c3', alpha=0.8)
axes[1].set_xlabel('N isolates')
axes[1].set_title('Sample size by country')
axes[1].set_xscale('log')

plt.suptitle('Country ranking (sorted by censoring rate, low to high)', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Continent-Level Split — Would Separate Models Help?

Check if different continents have fundamentally different MIC distributions (would warrant training separate models per region).

In [ ]:
from scipy import stats

continents = [c for c in df['continent'].unique() if c not in ('Other', 'Unknown')]
pairs = [(a, b) for i, a in enumerate(continents) for b in continents[i+1:]]

kw_stat, kw_p = stats.kruskal(*[df[df['continent'] == c]['log2_mic'].values
                                  for c in continents])
print(f'Kruskal-Wallis across continents: H={kw_stat:.1f}, p={kw_p:.2e}')
if kw_p < 0.05:
    print('=> MIC distributions differ significantly across continents.')
    print('   Separate continent models could capture region-specific resistance patterns.')
else:
    print('=> No significant difference. Continent splitting unlikely to help.')

# Continent-level summary table
cont_summary = df.groupby('continent').agg(
    n=('log2_mic', 'count'),
    cens_rate=('is_censored', 'mean'),
    mic_mean=('log2_mic', 'mean'),
    mic_p90=('log2_mic', lambda x: np.percentile(x, 90)),
    mic_iqr=('log2_mic', lambda x: np.percentile(x, 75) - np.percentile(x, 25)),
    resistant_pct=('resistant', 'mean'),
).round(3)

display(cont_summary.sort_values('resistant_pct', ascending=False))

In [ ]:
# Year trend per continent — slope of MIC90 over time
from numpy.polynomial import polynomial as P

print('MIC90 year slope by continent (log2 units/yr):')
print(f"  {'Continent':<25} {'slope (log2/yr)':>16} {'fold-change/yr':>15}")
print('  ' + '-' * 58)
for cont in sorted(continents):
    sub = df[df['continent'] == cont].groupby('year')['log2_mic'].quantile(0.9)
    if len(sub) < 5:
        continue
    slope, intercept = np.polyfit(sub.index, sub.values, 1)
    fc = 2**abs(slope) - 1
    direction = 'up' if slope > 0 else 'down'
    print(f"  {cont:<25} {slope:>+16.4f}  {fc:>13.1%}/yr {direction}")

## 10. Recommended Filter Candidates

Summary of what the exploration suggests.

In [ ]:
# Countries to consider dropping (very high censoring + few resistant obs)
drop_candidates = ctry_stats[
    (ctry_stats['censoring_rate'] > 0.90) &
    (ctry_stats['resistant_pct'] < 0.05)
].sort_values('censoring_rate', ascending=False)

print('Drop candidates (>90% censored AND <5% resistant):')
print(drop_candidates[['country', 'n', 'censoring_rate', 'resistant_pct']]
      .to_string(index=False))

print('\n')

# High-information countries (low censoring OR high resistant %)
keep_priority = ctry_stats[
    (ctry_stats['censoring_rate'] < 0.40) |
    (ctry_stats['resistant_pct'] > 0.20)
].sort_values('resistant_pct', ascending=False)

print('High-information countries (<40% censored OR >20% resistant):')
print(keep_priority[['country', 'continent', 'n', 'censoring_rate',
                     'resistant_pct', 'mic_iqr']]
      .head(20).to_string(index=False))

In [ ]:
# Final recommendation printout
n_drop = len(drop_candidates)
n_dropped_isolates = drop_candidates['n'].sum()
n_total = len(df)

print('=== FILTERING RECOMMENDATION ===')
print(f'''
Option A - Light filter (drop >90% censored + <5% resistant):
  Drop {n_drop} countries, lose {n_dropped_isolates:,} isolates ({n_dropped_isolates/n_total:.1%} of train set)
  Expected benefit: cleaner MIC distribution, less floor artifact

Option B - Strict filter (drop >75% censored countries):
  See summary table in section 7
  Higher data loss but significantly more diverse MIC values

Option C - Continent stratification:
  Train separate models per continent (or High/Low resistance tier)
  Best if Kruskal-Wallis above was significant
  Downside: small-continent models may have too few resistant obs

Option D - No filter, add continent as grouped feature:
  Replace 64-80 sparse country OHE with 6-7 continent columns
  Less sparsity, better generalization
  Implement in feature engineer.py before retraining XGBoost
''')